In [1]:
!pip install requests pandas

In [15]:
import requests
import pandas as pd

# API endpoint with OData filter for years >= 2025
#https://cbpfapi.unocha.org/vo1
#https://cbpf.data.unocha.org/dataexplorer.html
#https://gms.unocha.org/en

url = "https://cbpfapi.unocha.org/vo1/odata/ProjectSummaryWithLocationAndCluster?poolfundAbbrv=SUD15"

print("Fetching data from API...")
response = requests.get(url)

if response.status_code == 200:
    print("Data retrieved successfully!")
    
    data = response.json()
    records = data['value'] if 'value' in data else data
    
    df = pd.DataFrame(records)
    
    output_file = 'project_summary.csv'
    df.to_csv(output_file, index=False)
    
    print(f"Data saved to {output_file}")
    print(f"Total records: {len(df)}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nFirst few rows:")
    print(df.head())
    
else:
    print(f"Error: {response.status_code}")
    print(response.text)


Fetching data from API...
Data retrieved successfully!
Data saved to project_summary.csv
Total records: 7947

Columns: ['PooledFundId', 'PooledFundName', 'ChfId', 'ChfProjectCode', 'ExternalProjectCode', 'AllocationType', 'AllocationYear', 'AllocationSourceID', 'AllocationSourceName', 'OrganizationName', 'OrganizationType', 'ProjectTitle', 'ProjectDuration', 'Budget', 'EnvironmentMarker', 'GenderMarker', 'ActualStartDate', 'ActualEndDate', 'ProjectStatus', 'ProjectStatusId', 'ProjectStatusCode', 'ProcessStatus', 'ProcessStatusId', 'PartnerCode', 'Cluster', 'ClusterPercentage', 'AdminLocation1TypeName', 'AdminLocation1', 'AdminLocation1Latitude', 'AdminLocation1Longitude', 'AdminLevel1PCode', 'AdminLevel1Percentage']

First few rows:
   PooledFundId PooledFundName  ChfId        ChfProjectCode  \
0            15          Sudan     12  SUD-13/S1/WASH/UN/12   
1            15          Sudan     12  SUD-13/S1/WASH/UN/12   
2            15          Sudan    164    SUD-13/ER/N/UN/164   
3    

In [1]:
import requests

# Download Sudan Admin Level 1 (States) GeoJSON
url = "https://data.humdata.org/dataset/a0528526-5e29-42e6-a442-8fdbbee27834/resource/5c7b6e6c-2a32-49ec-8256-f8e4b3e907a1/download/sdn_admbnda_adm1_cbs_nic_ssa_20200831.geojson"

response = requests.get(url)
with open('sudan_admin1.geojson', 'wb') as f:
    f.write(response.content)

print("Sudan GeoJSON downloaded successfully!")


Sudan GeoJSON downloaded successfully!


In [9]:
import requests
import pandas as pd
import re

url = "https://cbpfapi.unocha.org/vo3/odata/GlobalGenericDataExtract?SPCode=CBPF_Global_PROJ_SUMMARY_Agg_V3&PoolfundCodeAbbrv=SUD15"

response = requests.get(url)
data = response.json()
records = data['value'] if 'value' in data else data

df = pd.DataFrame(records)

# -----------------------
# Function to sum beneficiaries
# -----------------------
def sum_beneficiaries(val):
    if pd.isna(val):
        return 0
    
    parts = re.split(r'##|\|\|\|', str(val))
    
    numbers = []
    for p in parts:
        p = p.strip()
        if p != "":
            try:
                numbers.append(float(p))
            except:
                pass
    
    return sum(numbers)

# -----------------------
# Create total columns
# -----------------------
df["Total_Planned_Admin1"] = df["AdmLocBenClustAgg1"].apply(sum_beneficiaries)
df["Total_Planned_Admin2"] = df["AdmLocBenClustAgg2"].apply(sum_beneficiaries)

df["Total_Reached_Admin1"] = df["AdmLocBenReachedClustAgg1"].apply(sum_beneficiaries)
df["Total_Reached_Admin2"] = df["AdmLocBenReachedClustAgg2"].apply(sum_beneficiaries)

# -----------------------
# Split coordinates
# -----------------------
if "AdmLocCord1" in df.columns:
    df[["Lat_Admin1", "Lon_Admin1"]] = df["AdmLocCord1"].str.split(",", expand=True)

if "AdmLocCord2" in df.columns:
    df[["Lat_Admin2", "Lon_Admin2"]] = df["AdmLocCord2"].str.split(",", expand=True)

# Convert to numeric
df["Lat_Admin1"] = pd.to_numeric(df["Lat_Admin1"], errors="coerce")
df["Lon_Admin1"] = pd.to_numeric(df["Lon_Admin1"], errors="coerce")

df["Lat_Admin2"] = pd.to_numeric(df["Lat_Admin2"], errors="coerce")
df["Lon_Admin2"] = pd.to_numeric(df["Lon_Admin2"], errors="coerce")

# -----------------------
# Drop old columns
# -----------------------
df.drop(columns=[
    "AdmLocBenClustAgg1",
    "AdmLocBenClustAgg2",
    "AdmLocBenReachedClustAgg1",
    "AdmLocBenReachedClustAgg2",
    "AdmLocCord1",
    "AdmLocCord2"
], inplace=True)

# -----------------------
# Show all columns
# -----------------------
pd.set_option('display.max_columns', None)

print(df.head())

# -----------------------
# Save cleaned dataset
# -----------------------
df.to_csv("project_summary_clean_totals_coords.csv", index=False)

   PFId                   PrjCode ClstAgg ClstPrct AdmLocTypeIdAgg  \
0    15  CBPF-SUD-23-R-INGO-24811      11   100.00        69##70##   
1    15  CBPF-SUD-23-R-INGO-24822      11   100.00        69##70##   
2    15  CBPF-SUD-23-R-INGO-24841      12   100.00        69##70##   
3    15  CBPF-SUD-23-R-INGO-24841      12   100.00        69##70##   
4    15  CBPF-SUD-23-R-INGO-24841      12   100.00        69##70##   

          AdmLoc1      AdmLoc2 AdmLocClustBdg1 AdmLocClustBdg2   AYr  \
0      White Nile  El Jabaleen       499998.57       499998.57  2023   
1     West Darfur   El Geneina       495000.00       495000.00  2023   
2       Blue Nile    El Kurmuk        10500.00        10500.00  2023   
3  South Kordofan     Al Buram        22050.00        22050.00  2023   
4  South Kordofan     AL Sunut         9000.00         9000.00  2023   

   Total_Planned_Admin1  Total_Planned_Admin2  Total_Reached_Admin1  \
0               14040.0               14040.0               14000.0   
1   

In [21]:
import requests
import pandas as pd
import re

# =============================================================================
# Join Two CBPF Datasets by Project Code
# =============================================================================
# Dataset 1 key: ChfProjectCode
# Dataset 2 key: PrjCode
# =============================================================================

KEY1 = "ChfProjectCode"  # Join key in Dataset 1
KEY2 = "PrjCode"         # Join key in Dataset 2

# -----------------------
# DATASET 1: Project Summary with Location and Cluster
# -----------------------
print("=" * 60)
print("Fetching Dataset 1: Project Summary with Location and Cluster")
print("=" * 60)

url1 = "https://cbpfapi.unocha.org/vo1/odata/ProjectSummaryWithLocationAndCluster?poolfundAbbrv=SUD15"

response1 = requests.get(url1)

if response1.status_code == 200:
    print("Dataset 1 retrieved successfully!")
    data1 = response1.json()
    records1 = data1['value'] if 'value' in data1 else data1
    df1 = pd.DataFrame(records1)
    print(f"Total records: {len(df1)}")
    print(f"Columns: {list(df1.columns)}\n")
else:
    print(f"Error fetching Dataset 1: {response1.status_code}")
    print(response1.text)
    df1 = pd.DataFrame()

# -----------------------
# DATASET 2: Global Generic Data Extract (Aggregated Project Summary)
# -----------------------
print("=" * 60)
print("Fetching Dataset 2: Global Generic Data Extract")
print("=" * 60)

url2 = "https://cbpfapi.unocha.org/vo3/odata/GlobalGenericDataExtract?SPCode=CBPF_Global_PROJ_SUMMARY_Agg_V3&PoolfundCodeAbbrv=SUD15"

response2 = requests.get(url2)

if response2.status_code == 200:
    print("Dataset 2 retrieved successfully!")
    data2 = response2.json()
    records2 = data2['value'] if 'value' in data2 else data2
    df2 = pd.DataFrame(records2)
    print(f"Total records: {len(df2)}")
    print(f"Columns: {list(df2.columns)}\n")
else:
    print(f"Error fetching Dataset 2: {response2.status_code}")
    print(response2.text)
    df2 = pd.DataFrame()

# -----------------------
# Function to sum beneficiaries from aggregated strings
# -----------------------
def sum_beneficiaries(val):
    """
    Parses aggregated beneficiary strings separated by ## or |||
    and returns the sum of all numeric values found.
    """
    if pd.isna(val):
        return 0

    parts = re.split(r'##|\|\|\|', str(val))

    numbers = []
    for p in parts:
        p = p.strip()
        if p != "":
            try:
                numbers.append(float(p))
            except ValueError:
                pass

    return sum(numbers)

# -----------------------
# Process Dataset 2: Create total beneficiary columns & coordinates
# -----------------------
print("=" * 60)
print("Processing Dataset 2: Computing totals and coordinates")
print("=" * 60)

if not df2.empty:

    # Sum beneficiary columns
    df2["Total_Planned_Admin1"] = df2["AdmLocBenClustAgg1"].apply(sum_beneficiaries)
    df2["Total_Planned_Admin2"] = df2["AdmLocBenClustAgg2"].apply(sum_beneficiaries)
    df2["Total_Reached_Admin1"] = df2["AdmLocBenReachedClustAgg1"].apply(sum_beneficiaries)
    df2["Total_Reached_Admin2"] = df2["AdmLocBenReachedClustAgg2"].apply(sum_beneficiaries)

    # Split coordinates
    if "AdmLocCord1" in df2.columns:
        df2[["Lat_Admin1", "Lon_Admin1"]] = df2["AdmLocCord1"].str.split(",", expand=True)
    if "AdmLocCord2" in df2.columns:
        df2[["Lat_Admin2", "Lon_Admin2"]] = df2["AdmLocCord2"].str.split(",", expand=True)

    # Convert coordinates to numeric
    for col in ["Lat_Admin1", "Lon_Admin1", "Lat_Admin2", "Lon_Admin2"]:
        if col in df2.columns:
            df2[col] = pd.to_numeric(df2[col], errors="coerce")

    # Drop raw aggregated columns
    cols_to_drop = [
        "AdmLocBenClustAgg1", "AdmLocBenClustAgg2",
        "AdmLocBenReachedClustAgg1", "AdmLocBenReachedClustAgg2",
        "AdmLocCord1", "AdmLocCord2"
    ]
    df2.drop(columns=[c for c in cols_to_drop if c in df2.columns], inplace=True)

    print(f"Dataset 2 processed. Records: {len(df2)}\n")

# -----------------------
# Join the two datasets on project code
# -----------------------
print("=" * 60)
print(f"Joining on: df1['{KEY1}'] <-> df2['{KEY2}']")
print("=" * 60)

if not df1.empty and not df2.empty:

    # Normalize key columns to string and strip whitespace
    df1[KEY1] = df1[KEY1].astype(str).str.strip()
    df2[KEY2] = df2[KEY2].astype(str).str.strip()

    # Rename KEY2 in df2 to match KEY1 so the merged column is consistent
    df2_renamed = df2.rename(columns={KEY2: KEY1})

    # Left join: keep all rows from Dataset 1, enrich with Dataset 2
    merged_df = pd.merge(df1, df2_renamed, on=KEY1, how="left")

    print(f"Merge complete!")
    print(f"  Dataset 1 rows : {len(df1)}")
    print(f"  Dataset 2 rows : {len(df2)}")
    print(f"  Merged rows    : {len(merged_df)}")
    print(f"  Merged columns : {len(merged_df.columns)}")
    print(f"\nMerged columns: {list(merged_df.columns)}")
    print(f"\nFirst few rows:")
    pd.set_option('display.max_columns', None)
    print(merged_df.head())

    # Save to CSV
    output_file = "project_summary_v2.csv"
    merged_df.to_csv(output_file, index=False)
    print(f"\nMerged data saved to '{output_file}'")

else:
    print("Join skipped — one or both datasets are empty.")


Fetching Dataset 1: Project Summary with Location and Cluster
Dataset 1 retrieved successfully!
Total records: 7947
Columns: ['PooledFundId', 'PooledFundName', 'ChfId', 'ChfProjectCode', 'ExternalProjectCode', 'AllocationType', 'AllocationYear', 'AllocationSourceID', 'AllocationSourceName', 'OrganizationName', 'OrganizationType', 'ProjectTitle', 'ProjectDuration', 'Budget', 'EnvironmentMarker', 'GenderMarker', 'ActualStartDate', 'ActualEndDate', 'ProjectStatus', 'ProjectStatusId', 'ProjectStatusCode', 'ProcessStatus', 'ProcessStatusId', 'PartnerCode', 'Cluster', 'ClusterPercentage', 'AdminLocation1TypeName', 'AdminLocation1', 'AdminLocation1Latitude', 'AdminLocation1Longitude', 'AdminLevel1PCode', 'AdminLevel1Percentage']

Fetching Dataset 2: Global Generic Data Extract
Dataset 2 retrieved successfully!
Total records: 1104
Columns: ['PFId', 'PrjCode', 'ClstAgg', 'ClstPrct', 'AdmLocTypeIdAgg', 'AdmLoc1', 'AdmLoc2', 'AdmLocBenClustAgg1', 'AdmLocBenClustAgg2', 'AdmLocCord1', 'AdmLocCord2'